### Extraction with head, and head, but limit the token to match the length of the method body

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import pandas as pd
import numpy as np
import tqdm as notebook_tqdm
from pathlib import Path
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import random
import os

In [ ]:

import pandas as pd

In [ ]:
### run 1, we will be using the llama3 model
checkpoint = "meta-llama/Llama-3.1-8B"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
device = 'cuda'
# for fp16 use `torch_dtype=torch.float16` instead
model = AutoModelForCausalLM.from_pretrained(checkpoint, device_map=device,torch_dtype=torch.float16)

In [ ]:
### run 2, we will be using the SC-7b model
checkpoint = "bigcode/starcoder2-7b"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
device = 'cuda'
# for fp16 use `torch_dtype=torch.float16` instead
model = AutoModelForCausalLM.from_pretrained(checkpoint, device_map=device, torch_dtype=torch.float16)

In [ ]:
df = pd.read_csv("/home/anonymous_user/Desktop/0423/input/step3_starcoder_7bpreceding_context_input.csv")

In [ ]:
df.head()

In [ ]:
df = df[df['preceding_context']!="Not found"]

In [ ]:
df = df[df['separated_method_snippet']!="Not found"]

In [ ]:
df.shape

In [ ]:
print(df.iloc[1]['method_head'])

In [ ]:
df[['method_head', 'method_body']] = df['separated_method_snippet'].str.extract(
    r'(?s)<first part>:(.*?)<second part>:(.*)'
)

In [ ]:
df['context_and_method_head'] = df['preceding_context'] + df['method_head']

run3: 50 input, min50/token length


In [ ]:
### Create a function that takes the last 50 tokens and makes it the input
def tokenize_and_select_last_50(df, tokenizer, device):
    for index, row in df.iterrows():
        content = row['context_and_method_head']
        full_encoded = tokenizer(content, return_tensors='pt').to(device)
        full_tokens = tokenizer.convert_ids_to_tokens(full_encoded.input_ids[0])
        last_50_tokens = full_tokens[-50:]
        last_50_snippet = tokenizer.convert_tokens_to_string(last_50_tokens)
        df.at[index, 'last_50_tokens_snippet'] = last_50_snippet
    return df


### Measure the actual token length of the method body and store it
def measure_method_body_length(df, tokenizer, device):
    for index, row in df.iterrows():
        full_ref = row['method_body']
        full_encoded = tokenizer(full_ref, return_tensors="pt").to(device)
        full_tokens = tokenizer.convert_ids_to_tokens(full_encoded.input_ids[0])
        method_body_length = len(full_tokens)  # Measure token length
        df.at[index, 'method_body_length'] = method_body_length  # Store in DataFrame
    return df


### Create a function that takes the first 50 tokens or the full method body if shorter
def create_ref(df, tokenizer, device):
    for index, row in df.iterrows():
        full_ref = row['method_body']
        full_encoded = tokenizer(full_ref, return_tensors="pt").to(device)
        full_tokens = tokenizer.convert_ids_to_tokens(full_encoded.input_ids[0])
        method_body_length = len(full_tokens)
        ref_tokens = full_tokens[:min(50, method_body_length)]  # Take first 50 or full length
        ref_string = tokenizer.convert_tokens_to_string(ref_tokens)
        df.at[index, 'reference_llama'] = ref_string
    return df


### Create a function that calculates the BLEU score for 50 tokens
def bleu_50(result_df):
    try:
        for index, row in result_df.iterrows():
            reference = row['reference_llama'].split()
            result = row['inference_llama'].split()
            smooth_fn = SmoothingFunction().method1
            bleu_score = sentence_bleu(
                [reference], result, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth_fn
            )
            result_df.at[index, 'code_only_bleu_llama'] = bleu_score
        return result_df
    except Exception as e:
        print(f"Error in bleu_50: {e}")
        return result_df


### Run inference, generate actual length or 50 tokens, whichever is shorter
def inference_50(df, model, tokenizer, device):
    for index, row in df.iterrows():
        inference_50_inputs = tokenizer.encode(row['last_50_tokens_snippet'], return_tensors="pt").to(device)
        actual_length = int(row['method_body_length'])  # Get stored method body length
        max_tokens = max(1, min(50, actual_length))  # Generate the actual length or 50, whichever is smaller
        inference_50_outputs = model.generate(inference_50_inputs, max_new_tokens=max_tokens)
        inference_generated_tokens = inference_50_outputs[:, inference_50_inputs.shape[1]:]
        inference_generated_result = tokenizer.decode(inference_generated_tokens[0], skip_special_tokens=True)
        df.at[index, 'inference_llama'] = inference_generated_result
    return df

In [ ]:
def process_dataframe(df, model, tokenizer, device):
    df = tokenize_and_select_last_50(df, tokenizer, device)
    df = measure_method_body_length(df, tokenizer, device)
    df = create_ref(df, tokenizer, device)
    df = inference_50(df, model, tokenizer, device)
    df = bleu_50(df)
    return df

In [ ]:
df = df.dropna()

In [ ]:
df.isna().sum()

In [ ]:
df_no_code_na = df.dropna(subset=['code'])

In [ ]:
df = df.dropna(subset=['comment'])

In [ ]:
df.isnull().sum()

In [ ]:
df_no_code_na.isnull().sum()

In [ ]:
df_no_code_na.shape

In [ ]:
df_result = process_dataframe(df_no_code_na, model, tokenizer, device)

In [ ]:
df_result['code_only_bleu_llama'].mean()

In [ ]:
df_result.to_csv("/home/anonymous_user/Desktop/0423/output/step3_output_starcoder_7b_50_tokens_bleu.csv", index=False)

In [ ]:
df_result.to_csv("/media/xiao/Lexar/0420/step_3_result_llama_50input_limited_output_run4023.csv")

In [ ]:
df = pd.read_csv("/media/xiao/Lexar/0423/output/step_2_result_llama.csv")

In [ ]:
df.head()

In [ ]:
df_no_code_na = pd.read_csv("/media/xiao/Lexar/0423/output/50tokeninput_output/step_3_result_llama_50input_limited_output_run4023.csv")

In [ ]:
### create a function that takes the last 150 tokens and make it the input

def tokenize_and_select_last_150(df, tokenizer, device):
    for index, row in df.iterrows():
        content = row['context_and_method_head']
        full_encoded = tokenizer(content, return_tensors='pt').to(device)
        full_tokens = tokenizer.convert_ids_to_tokens(full_encoded.input_ids[0])
        last_150_tokens = full_tokens[-150:]
        last_150_snippet = tokenizer.convert_tokens_to_string(last_150_tokens)
        df.at[index, 'last_150_tokens_snippet'] = last_150_snippet
    return df


In [ ]:
df_no_code_na.head()

In [ ]:
df_no_code_na = tokenize_and_select_last_150(df_no_code_na, tokenizer, device)

In [ ]:
### run the inference, 3b, full point, 50 tokens
def inference_50_w_150(df, model, tokenizer, device):
    for index, row in df.iterrows():
        inference_150_inputs =tokenizer.encode(row['last_150_tokens_snippet'], return_tensors="pt").to(device)
        actual_length = int(row['method_body_length'])  # Get stored method body length
        max_tokens = max(1, min(50, actual_length))  # Generate the actual length or 50, whichever is smaller
        inference_150_outputs = model.generate(inference_150_inputs, max_tokens)
        inference_generated_tokens = inference_150_outputs[:, inference_150_inputs.shape[1]:]
        inference_generated_result = tokenizer.decode(inference_generated_tokens[0], skip_special_tokens=True)
        df.at[index, 'inference_50_w_150'] = inference_generated_result
    return df

In [ ]:
df_no_code_na = inference_50_w_150(df_no_code_na, model, tokenizer, device)

In [ ]:
def inference_50_w_150(df, model, tokenizer, device):
    # make sure pad_token_id is defined for the model
    model.config.pad_token_id = tokenizer.eos_token_id

    for index, row in df.iterrows():
        # encode the prompt
        inputs = tokenizer(
            row['last_150_tokens_snippet'],
            return_tensors="pt"
        ).input_ids.to(device)

        # clamp to [1, 50]
        actual_length = int(row['method_body_length'])
        max_new = max(1, min(50, actual_length))

        # generate with the proper kwarg
        outputs = model.generate(
            inputs,
            max_new_tokens=max_new,
            pad_token_id=tokenizer.eos_token_id
        )

        # strip off the prompt tokens
        gen_tokens = outputs[:, inputs.shape[1]:]
        gen_text   = tokenizer.decode(gen_tokens[0], skip_special_tokens=True)

        df.at[index, 'inference_50_w_150'] = gen_text

    return df

In [ ]:
df_no_code_na = inference_50_w_150(df_no_code_na, model, tokenizer, device)

In [ ]:

### create a function that takes the 50 token result and calculates the bleu score
def bleu_50_w_150(result_df):
    try:
        for index, row in result_df.iterrows():
            reference = row['reference_llama'].split()
            result = row['inference_50_w_150'].split()
            smooth_fn = SmoothingFunction().method1
            bleu_score = sentence_bleu([reference], result, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth_fn)
            result_df.at[index, 'code_only_bleu_150'] = bleu_score
        return result_df
    except Exception as e:
        print(f"Error in four_gram_bleu: {e}")
        return result_df

In [ ]:
df_no_code_na = bleu_50_w_150(df_no_code_na)

In [ ]:
df_no_code_na['code_only_bleu_150'].mean()


In [ ]:
df_no_code_na.to_csv("/media/xiao/Lexar/0423/output/150tokeninput_output/step3_result_150_input_llama.csv", index=False)